# Automação da Busca de Livros

Este notebook foi desenvolvido para automatizar a busca de livros em diferentes sites.

Primeiro, os livros são carregados a partir de uma planilha. Em seguida, cada livro é pesquisado no Gutenberg e, caso não seja encontrado, a busca é realizada no Books to Scrape.

Ao final do processo, os preços e links encontrados são registrados automaticamente na planilha.

In [ ]:
# Importa as bibliotecas utilizadas
from selenium import webdriver
from selenium.webdriver.common.keys import Keys

# Abre o navegador
navegador = webdriver.Chrome()

## Carregamento dos livros

Nesta etapa, a planilha com os livros é carregada para que os dados possam ser utilizados durante as pesquisas.

A base contém as informações necessárias para realizar a busca, como nome, autor e categoria dos livros.

In [ ]:
# Carrega a lista de livros
import pandas as pd
df_produtos = pd.read_excel("Produtos.xlsx")

# Exibe os livros carregados
print(df_produtos)

## Pesquisa dos livros

Nesta etapa, cada livro é pesquisado automaticamente.

A busca é realizada primeiro no Gutenberg. Caso o livro não seja encontrado, o processo continua no Books to Scrape.

Ao encontrar o livro, são registrados o link e o preço disponíveis no site.

In [ ]:
# Pesquisa os livros no Gutenberg

def pesquisar_gutenberg(nome, autor, navegador):

    lista_palavras_autor = autor.split(" ")

    navegador.get("https://www.gutenberg.org/")

    elemento = navegador.find_element(
        "class name",
        "search-input"
    )

    elemento.send_keys(nome)
    elemento.send_keys(Keys.ENTER)

    lista_resultados = navegador.find_elements(
        "class name",
        "booklink"
    )

    link = None


    # Verifica os resultados encontrados

    for resultado in lista_resultados:

        texto = resultado.text.lower()

        if nome.lower() in texto:

            if all(
                palavra.lower() in texto
                for palavra in lista_palavras_autor
            ):

                link = resultado.find_element(
                    "class name",
                    "link"
                ).get_attribute("href")

                break


    # Define o preço do livro

    if link:
        preco = 0
    else:
        preco = None

    return link, preco


# Pesquisa os livros no Books to Scrape

def pesquisar_bookstocrape(nome, autor, navegador):

    navegador.get("https://books.toscrape.com/")

    link = None
    preco = None
    encontrado = False

    while not encontrado:

        # Lista os livros da página atual

        lista_resultados = navegador.find_elements(
            "class name",
            "product_pod"
        )

        # Verifica os livros encontrados

        for resultado in lista_resultados:

            elemento_h3 = resultado.find_element(
                "tag name",
                "h3"
            )

            elemento_do_link = elemento_h3.find_element(
                "tag name",
                "a"
            )

            titulo = elemento_do_link.get_attribute(
                "title"
            )

            if nome.lower() in titulo.lower():

                encontrado = True

                # Pega o link do livro

                link = elemento_do_link.get_attribute(
                    "href"
                )

                # Pega o preço do livro

                preco = resultado.find_element(
                    "class name",
                    "price_color"
                ).text

                break


        # Encerra a busca caso o livro seja encontrado

        if encontrado:
            break

        # Tenta acessar a próxima página

        try:

            navegador.find_element(
                "link text",
                "next"
            ).click()

        except:

            break

    return link, preco

### Preparação da planilha

Antes de iniciar as pesquisas, as colunas de link e preço são configuradas para aceitar os valores encontrados durante o processo.

Em seguida, cada livro da planilha é pesquisado automaticamente.

In [ ]:
# Corrige o tipo das colunas

df_produtos["link"] = df_produtos["link"].astype("object")
df_produtos["preco"] = df_produtos["preco"].astype("object")


# Percorre cada livro da planilha

for linha in df_produtos.index:

    nome = df_produtos.loc[linha, "nome"]
    autor = df_produtos.loc[linha, "autor"]
    categoria = df_produtos.loc[linha, "categoria"]


    # Pesquisa primeiro no Gutenberg

    link1, preco1 = pesquisar_gutenberg(
        nome,
        autor,
        navegador
    )

    print(nome, link1, preco1)


    # Se não encontrou no Gutenberg, pesquisa no Books to Scrape

    if not link1:

        link2, preco2 = pesquisar_bookstocrape(
            nome,
            autor,
            navegador
        )

        print(nome, link2, preco2)

        df_produtos.loc[linha, "preco"] = preco2
        df_produtos.loc[linha, "link"] = link2


    # Registra os dados encontrados no Gutenberg

    else:

        df_produtos.loc[linha, "preco"] = preco1
        df_produtos.loc[linha, "link"] = link1
        

## Atualização da planilha

Após finalizar a pesquisa de todos os livros, os resultados são salvos na planilha.

Ao final, o navegador é encerrado para finalizar a automação.

In [ ]:
# Salva os resultados na planilha
df_produtos.to_excel("Produtos.xlsx", index=False)

# Encerra o navegador
navegador.quit()
